In [1]:
# Conversion Service
from opioid import Opioid
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS

conversion_service = ConversionService(NIELSON_CONVERSIONS)

iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone")
oral_morphine = Opioid(administration_method="oral", substance="morphine")

random_dose = conversion_service.generate_random_dose(iv_fentanyl)
print(f"Random dose for {iv_fentanyl.administration_method}_{iv_fentanyl.substance}: {random_dose:.2f} {iv_fentanyl.unit}")

equivalent_dose_oral_morphine = conversion_service.calculate_expected_dose(random_dose, iv_fentanyl, oral_morphine)
print(f"Equivalent dose for {oral_morphine.administration_method}_{oral_morphine.substance}: {equivalent_dose_oral_morphine:.2f} {oral_morphine.unit}")

Random dose for iv_fentanyl: 0.11 mcg/hour
Equivalent dose for oral_morphine: 11.04 mg/day


In [1]:
# Prompt Service
from prompt_service import PromptService, PromptedOpioidConversion
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS
from opioid import Opioid

# Initialize the PromptService
prompt_service = PromptService(ConversionService(NIELSON_CONVERSIONS))

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/hour")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/day")

# Create a PromptedOpioidConversion object without specifying `from_dose` or `conversion_service`
prompted_conversion = PromptedOpioidConversion(
    from_opioid=iv_fentanyl,
    model="gpt-4.1-mini",)

# Generate the prompt using the PromptService instance
generated_prompt = prompted_conversion.generate_prompt(prompt_service)

# Print the generated prompt
print("Generated Prompt:")
print(generated_prompt)

# Print the randomly generated `from_dose` and calculated `expected_to_dose`
print(f"\nRandomly Generated Dose: {prompted_conversion.from_dose:.2f} {iv_fentanyl.unit}")
print(f"Expected Dose: {prompted_conversion.expected_to_dose:.2f} {oral_hydromorphone.unit}")

Generated Prompt:
Convert 0.13 mcg/hour of fentanyl (iv) to mg/day of morphine (oral).
How many mg/day of morphine (oral) is equivalent to 0.13 mcg/hour of fentanyl (iv)?

Use the following conversion factor(s):
100 OME: 1 mcg/hour of fentanyl (iv) is equivalent to 100 mg/day morphine (oral).
100 OME for fentanyl (iv) means 10 mg/day of morphine (oral) is equivalent to 0.10 mcg/hour of fentanyl (iv).
And 120 mg/day of morphine (oral) is equivalent to 1.20 mcg/hour of fentanyl (iv).

Express the response as a single numeric value without any additional text in the format {calculated dose: value}.

Randomly Generated Dose: 0.13 mcg/hour
Expected Dose: 13.00 mg/day


In [ ]:
# Run a single conversion with an instance of PromptedOpioidConversion
from conversion_service import ConversionService
from prompt_service import PromptService, PromptedOpioidConversion
from nielson_data import NIELSON_CONVERSIONS
from opioid import Opioid

# Initialize the OpenAI client once
PromptedOpioidConversion.initialize_client()

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/hour")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/day")

# Create a PromptedOpioidConversion object
conversion = PromptedOpioidConversion(
    from_opioid=iv_fentanyl,
    include_nielson=True,
    include_examples=False
)

# Generate the prompt
prompt_service=PromptService(ConversionService(NIELSON_CONVERSIONS))
conversion.generate_prompt(prompt_service=prompt_service)

# Query the model
observed_dose = conversion.ask_model()

# Print the results
print("Generated Prompt:")
print(conversion.prompt)
print(f"Observed Dose: {observed_dose:.2f} {oral_hydromorphone.unit}")
print(f"Expected Dose: {conversion.expected_to_dose:.2f} {oral_hydromorphone.unit}")
print(f"Ratio Observed/Expected: {conversion.ratio_observed_expected:.2f}")

Generated Prompt:
Convert 0.32 mcg/hour of fentanyl (iv) to mg/day of morphine (oral).
How many mg/day of morphine (oral) is equivalent to 0.32 mcg/hour of fentanyl (iv)?

Use the following conversion factor(s):
100 OME: 1 mcg/hour of fentanyl (iv) is equivalent to 100 mg/day morphine (oral).

Express the response in the format {'calculated dose': value}.  Do not include any additional text in your response, only the dictionary.
Observed Dose: 32.00 mg/day
Expected Dose: 32.00 mg/day
Ratio Observed/Expected: 1.00


In [1]:
# Run an experiment
from prompt_service import PromptedOpioidConversion
from opioid import Opioid
from experiment import Experiment

# Initialize the OpenAI client once
PromptedOpioidConversion.initialize_client()

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/hour")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/day")
oral_morphine = Opioid(administration_method="oral", substance="morphine", unit="mg/day")

# Define opioid pairs for the experiment
opioid_pairs = [
    (iv_fentanyl, oral_morphine),
]

# Define prompt conditions
prompt_conditions = [
    {"include_nielson": False, "include_examples": False},
    {"include_nielson": True, "include_examples": False},
    {"include_nielson": False, "include_examples": True}
]

# Run an experiment
experiment = Experiment(opioid_pairs, prompt_conditions, repetitions=30)
experiment.run()
experiment.save_to_csv("experiment_results.csv")

# Print the experiment ID
print(f"Experiment ID: {experiment.experiment_id}")

Results appended to experiment_results.csv
Experiment ID: 10


In [ ]:
from experiment import ExperimentFromCSVConverter

# Define the prompt conditions to filter by
prompt_conditions = [
    {"include_nielson": False, "include_examples": False},
    {"include_nielson": True, "include_examples": False}
]

# Recreate the Experiment object with matching prompt_conditions
experiment_from_csv = ExperimentFromCSVConverter.from_csv(
    "experiment_results.csv",
    prompt_conditions=prompt_conditions
)

# Use the Experiment object as usual
experiment_from_csv.visualize()
aggregates = experiment_from_csv.calculate_aggregates()
print(aggregates)

TypeError: ExperimentFromCSVConverter.from_csv() got an unexpected keyword argument 'prompt_conditions'